# D3-04 Optional custom scenarios with premise - RTE FE2050

This optional notebook runs a real `premise` build that combines one IAM scenario with two French RTE `Futurs énergétiques 2050` custom scenarios.

It assumes that:
- the Day 3 Brightway project is `paris-lca-course-2026-premise`;
- `ecoinvent-3.12-cutoff` and `ecoinvent-3.12-biosphere` are available in that project;
- the course `.env` file contains `IAM_FILES_KEY`, or the embedded course key is acceptable;
- the RTE scenario package can be cloned from GitHub.


In [ ]:
import os
import subprocess
from pathlib import Path

import bw2data as bd
from datapackage import Package
from dotenv import load_dotenv
from premise import NewDatabase
load_dotenv()

project_name = "paris-lca-course-2026-premise"
source_db = "ecoinvent-3.12-cutoff" # adapt to the name of your database
biosphere_name = "biosphere-3.12" # adapt to the name of your database
source_version = "3.12"

iam_key = os.environ.get(
    "IAM_FILES_KEY",
    "tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=",
)


## Get the RTE scenario package

The repository is cloned outside the course repository to avoid creating files that could conflict with course updates.

Clone it, by doing:

```bash

git clone https://github.com/oie-mines-paristech/RTE_scenarios.git

```


Then, enter the clone repository, and switch to the ecoinvent3.12-compatible branch, by doing:

```bash
git checkout ecoinvent-3.12-cutoff
```

## Define the combined scenarios

The IAM scenario controls the global background trajectory. The RTE external scenario controls the French electricity, fuel, gas, and hydrogen assumptions.


In [ ]:
rte_dir = Path("/Users/romain/GitHub/RTE_scenarios") # filepatch to the clone repo on your computer
datapackage_path = rte_dir / "datapackage.json"
rte = Package(str(datapackage_path))


model = "tiam-ucl"
pathway = "SSP2-RCP45"
year = 2050
french_scenarios = ["Reference - M0", "Reference - N03"]

scenarios = [
    {
        "model": model,
        "pathway": pathway,
        "year": year,
        "external scenarios": [{"scenario": french_scenario, "data": rte}],
    }
    for french_scenario in french_scenarios
]

for scenario in scenarios:
    print(scenario["model"], scenario["pathway"], scenario["year"], scenario["external scenarios"][0]["scenario"])


## Run premise and write the databases

This cell performs the real update and writes the resulting scenario databases to Brightway. It can take several minutes and may use several GB of memory.


In [ ]:
bd.databases


In [ ]:
bd.projects.set_current(project_name)

missing = [name for name in [source_db, biosphere_name] if name not in bd.databases]
if missing:
    raise ValueError(f"Missing Brightway databases in {project_name}: {missing}")

before_databases = set(bd.databases)

ndb = NewDatabase(
    scenarios=scenarios,
    source_db=source_db,
    source_version=source_version,
    source_type="brightway",
    system_model="cutoff",
    biosphere_name=biosphere_name,
    key=iam_key,
    keep_imports_uncertainty=True,
    keep_source_db_uncertainty=False,
)

ndb.update()
ndb.write_db_to_brightway()

new_databases = sorted(set(bd.databases) - before_databases)
print("New databases:")
for database_name in new_databases:
    print("-", database_name)


## Check the project databases


In [ ]:
list(bd.databases)


In [ ]:
import bw2calc as bc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_scenario_database(french_scenario):
    database_names = sorted(str(name) for name in bd.databases)
    candidates = [
        name
        for name in database_names
        if model in name and pathway in name and str(year) in name and french_scenario in name
    ]
    if not candidates:
        candidates = [
            name
            for name in database_names
            if french_scenario in name and str(year) in name
        ]
    if not candidates:
        raise ValueError(f"Could not find a Brightway database for scenario: {french_scenario}")
    return candidates[-1]


def find_low_voltage_activity(database_name):
    name_candidates = [
        "market for electricity, low voltage, FE2050",
        "market for electricity, low voltage",
    ]
    location_candidates = ["FR", "FRA"]
    database = bd.Database(database_name)

    for target_name in name_candidates:
        matches = [
            activity
            for activity in database
            if activity.get("name") == target_name
            and activity.get("reference product") == "electricity, low voltage"
            and activity.get("location") in location_candidates
        ]
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise ValueError(f"Found multiple matches for {target_name} in {database_name}: {matches}")

    details = [
        (activity.get("name"), activity.get("reference product"), activity.get("location"))
        for activity in database
        if "electricity" in activity.get("name", "")
        and "low voltage" in activity.get("name", "")
        and activity.get("location") in location_candidates
    ][:20]
    raise ValueError(f"Could not find a French low-voltage electricity market in {database_name}. Candidates: {details}")


def location_contributions(activity, method):
    lca = bc.LCA({activity.id: 1}, method)
    lca.lci()
    lca.lcia()

    contributions = np.asarray(lca.characterized_inventory.sum(axis=0)).ravel()
    rows = []
    for matrix_index, contribution in enumerate(contributions):
        if abs(contribution) < 1e-15:
            continue
        node_id = lca.dicts.activity.reversed[matrix_index]
        try:
            node = bd.get_node(id=node_id)
        except Exception:
            node = bd.get_activity(node_id)
        rows.append(
            {
                "location": node.get("location") or "unknown",
                "contribution": float(contribution),
            }
        )
    return rows, float(lca.score)


method = ("EF v3.1", "climate change", "global warming potential (GWP100)")
method_unit = bd.Method(method).metadata.get("unit", "score")
database_cases = [("ecoinvent 3.12 cutoff", source_db)] + [
    (french_scenario, find_scenario_database(french_scenario))
    for french_scenario in french_scenarios
]

rows = []
for case_label, database_name in database_cases:
    activity = find_low_voltage_activity(database_name)
    contribution_rows, score = location_contributions(activity, method)
    for row in contribution_rows:
        rows.append(
            {
                "case": case_label,
                "database": database_name,
                "score": score,
                **row,
            }
        )

contributions = pd.DataFrame(rows)
top_locations = (
    contributions.assign(abs_contribution=contributions["contribution"].abs())
    .groupby("location")["abs_contribution"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
)
contributions["location_group"] = np.where(
    contributions["location"].isin(top_locations),
    contributions["location"],
    "Other",
)

plot_data = (
    contributions.groupby(["case", "location_group"], sort=False)["contribution"]
    .sum()
    .unstack(fill_value=0)
    .reindex([label for label, _ in database_cases])
)
ordered_columns = [location for location in top_locations if location in plot_data.columns]
if "Other" in plot_data.columns:
    ordered_columns.append("Other")
plot_data = plot_data[ordered_columns]

ax = plot_data.plot(kind="bar", stacked=True, figsize=(9, 5), colormap="tab20")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel(method_unit)
ax.set_xlabel("")
ax.set_title("1 kWh French low-voltage electricity: location-based LCIA contributions")
ax.legend(title="Activity location", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

plot_data


In [ ]:
method in bd.methods